# 1. Create an acquisition and inspect the pulse

**Scenario:** you want to check what a dispersed fast radio burst (FRB) looks like before searching for it. You will generate a short, bright example with PIRATE, then inspect its saved intensity data.

Use your own PIRATE environment in a GPU allocation, following [SETUP_FIR.md](SETUP_FIR.md). Start your **Bash terminal and notebook kernel in this tutorial repository**, beside the notebooks and configs/. Select the environment checked in SETUP_FIR.md, even if another kernel is also named PIRATE.

The workflow is: **read the configuration -> copy the command into the terminal -> wait for completion -> run the Python plotting cells**. The notebook does not launch processing. The instructor has reported a successful local walkthrough. This standalone edition and installation on Fir still need validation.

## 1. Read the configuration

Open [configs/metadata.yml](configs/metadata.yml). This native file describes the data layout, not the injected pulse. Focus on:

~~~yaml
zone_freq_edges: [300, 350, 450, 600, 800, 1500]  # MHz
zone_nfreq: [8192, 8192, 6144, 2048, 3584]
beam_ids: [1]
dt_ns_per_seq: 5120
seq_per_frb_time_sample: 195
noise_variance: 1.0
~~~

The five frequency zones contain 28160 channels in total. Each zone has equal-width channels, but different zones have different channel widths. The two time fields give **5120 x 195 ns = 0.9984 ms per sample**. Beam 1 represents one pointing direction. Leave the required telescope-geometry fields at their supplied example values.

We will inject one pulse using command-line options:

| Option | Baseline value | Meaning |
| --- | --- | --- |
| --dm | 20 | Dispersion measure in pc cm^-3: the integrated free-electron density along the line of sight |
| --toa | 5 | Arrival time in seconds at the **lowest band edge, 300 MHz**, from observation start |
| --width-ms | 3 | Intrinsic Gaussian sigma (standard deviation), in ms |
| --snr | 200 | Integrated signal-to-noise ratio (S/N), not the S/N of one spectrogram pixel |
| --nchunks / --ntime | 4 / 2048 | Four chunks of 2048 samples: 8.1788928 s in total |

Low frequencies arrive later. For these settings, the dispersed pulse lies approximately between 4.11 and 5 s, entirely in chunk 2. The pulse is deliberately bright for this first exercise.

## 2. Choose your output folder

In the Bash block below, fill in **RUN_DIR** between the empty quotes. Use the full path to a **new folder inside an existing writable directory**, for example a new experiment folder in your personal scratch space. Leave no extra spaces inside the quotes. The same full path goes in **run_directory** in the Python cell and in notebooks 2 and 3.

This run folder is separate from the tutorial repository. The command will create it, copy the three configs, and save generation.log and a frames/ subfolder inside it. See the [folder layout](README.md#paths-and-output-layout).

Run the block from the tutorial repository so configs/ points to the files supplied here. Each step joined by **&&** runs only if the previous step succeeds. An empty or existing RUN_DIR makes mkdir fail, stopping the remaining steps without closing your terminal.


## 3. Generate the acquisition in the terminal

PIRATE's CLI module is **pirate_frb.make_simulated_acq**. Fill in RUN_DIR, then copy this block into your activated GPU terminal:

~~~bash
RUN_DIR=""  # REQUIRED: full path to a new run folder; its parent must already exist.
mkdir "$RUN_DIR" &&
cp configs/metadata.yml configs/dedispersion.yml configs/grouper.yml "$RUN_DIR/" &&
python -m pirate_frb.make_simulated_acq \
    "$RUN_DIR/metadata.yml" "$RUN_DIR/frames" \
    --nchunks 4 --ntime 2048 \
    --dm 20 --toa 5 --width-ms 3 --snr 200 \
    --sm 0 --spectral-index 0 \
    --subband-lo-MHz 300 --subband-hi-MHz 1500 \
    > "$RUN_DIR/generation.log" &&
cat "$RUN_DIR/generation.log"
~~~

The final options select no scattering, a flat spectral index, and emission across the full band. The summary is saved to generation.log and displayed when generation succeeds. Wait for **Finished: wrote 4 frames** and check that the pulse's observation status is **fully inside**. The generator reports clipping but does not reject a clipped pulse.

**Stop if any command fails.** If the run folder already exists, choose a new name; rerunning the generator alone can overwrite its frames. Keep generation.log with the data: it records the resolved pulse parameters, time reference, and sample range. Save the command you actually used alongside it when changing options.

**There is no seed option in this CLI.** Do not expect regenerated noise to match an earlier run. Reuse the same saved frames when comparing later processing settings.

Each frame_b1_tN.asdf contains one beam and one time chunk: quantized frequency-time intensity samples plus metadata. Quantization stores intensity with limited numerical precision. These are intensity data, not raw voltages or image files. The first chunk contains only noise; chunk 2 contains the pulse.


## 4. Read the saved files in Python

Only continue once the terminal command has completed successfully. These cells read the existing run; rerunning them does not regenerate it.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from pirate_frb.core import AssembledFrame


In [ ]:
run_directory = ""  # REQUIRED: full path to the run folder you chose in the terminal.
if not run_directory.strip():
    raise ValueError("Fill in run_directory above before continuing.")
run_dir = Path(run_directory).expanduser()
print((run_dir / 'generation.log').read_text())


## 5. Why do channel widths vary?

Dispersion changes the arrival time more rapidly at low frequency. A finite channel therefore blurs the pulse across its own frequency interval: **intra-channel dispersion smearing**.

For a fixed DM and a narrow channel, that smear is approximately proportional to channel width divided by frequency cubed. The supplied CHORD-like layout uses narrower channels at low frequency and wider ones at high frequency to keep the smear roughly comparable across the band. This is a piecewise approximation, **not exactly constant smearing in every channel**.

The figure below shows the actual stored channel widths. This physical channelization is distinct from the internal inverse-frequency-squared grid used by PIRATE's dedispersion tree.

In [ ]:
frame = AssembledFrame.from_asdf(str(run_dir / 'frames/frame_b1_t0.asdf'))
frequency_edges = np.asarray(frame.metadata.get_channel_freq_edges())
dt = frame.metadata.dt_ns_per_seq * frame.metadata.seq_per_frb_time_sample * 1e-9
ntime = frame.ntime

fig, ax = plt.subplots(figsize=(8, 3), constrained_layout=True)
ax.step(frequency_edges[:-1], np.diff(frequency_edges) * 1000, where='post')
ax.set(xlabel='Frequency (MHz)', ylabel='Channel width (kHz)',
       title='Physical frequency channels in the saved acquisition')
plt.show()
del frame

**What to look for:** channel widths increase in steps toward higher frequencies. The steps mark the five zones, not gaps in frequency coverage. Every channel inside a zone has the same width.

## 6. Display a noise interval and the pulse

PIRATE's native dequantize() reads the stored intensity values as floating-point data. We average blocks of 128 adjacent channels **only for display**; neither the saved frames nor the later search are changed. The plot uses the real frequency edges because these display bins also have unequal widths.

Only one full-resolution frame is expanded at a time; that temporary array alone uses about 220 MiB of host memory. The two panels contain **different noise samples**, not the same frame before and after injection.

In [ ]:
chunks = [0, 2]
spectrograms = []
for chunk in chunks:
    frame = AssembledFrame.from_asdf(str(run_dir / 'frames' / f'frame_b1_t{chunk}.asdf'))
    intensity = frame.dequantize()
    spectrograms.append(intensity.reshape(-1, 128, frame.ntime).mean(axis=1))
    del intensity
del frame
display_frequency_edges = frequency_edges[::128]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True, constrained_layout=True)
for ax, chunk, values, title in zip(
        axes, chunks, spectrograms, ['Noise-only interval', 'Interval containing the pulse']):
    time_edges = (chunk * ntime + np.arange(ntime + 1)) * dt
    image = ax.pcolormesh(time_edges, display_frequency_edges, values, shading='flat',
                         cmap='magma', vmin=-0.3, vmax=0.6, rasterized=True)
    ax.set(xlabel='Time since observation start (s)', title=title)
axes[0].set_ylabel('Frequency (MHz)')
fig.colorbar(image, ax=axes, label='Mean intensity (arbitrary units)')
plt.show()

**What to look for:** a curved track whose high-frequency part arrives first. Its low-frequency end reaches 300 MHz near 5 s. Both panels use the same color limits. The simulated samples are normalized intensity fluctuations around zero, so negative values are expected; they are not negative physical received power.

### Try one change

Record the baseline appearance, or save a notebook copy with the figures. In the terminal, repeat the generation block with **RUN_DIR set to a different, new folder** and **--snr 100**. Keep every other pulse option unchanged. Then set run_directory to that new folder in Python and rerun the reading/plotting cells.

Predict whether the track becomes harder to see. Its expected timing stays the same, but this is a new noise realization as well as a fainter pulse.

Continue with [notebook 2](02_dedispersion.ipynb), using the same run directory in both its terminal commands and plotting cells.